In [4]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import pandas as pd
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


train = pd.read_csv('train_cleaned.csv')
test = pd.read_csv('test_cleaned.csv')
feature_cols = [
    "Applied_Voltage_kV", "Load_Current_A", "Ambient_Temperature_C", "Test_Duration_min",
    "Sensor_S1", "Sensor_S2", "Sensor_S3",
    "S1_missing", "S2_missing", "S3_missing", "S4_missing",
    "is_duplicate_input"]


X = train[feature_cols]
y_class = train["Validity_Label_enc"]
y_reg = train["Reference_Parameter"]


lgb_f1_scores = []
lgb_auc_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_class.iloc[train_idx], y_class.iloc[val_idx]

    clf_lgb = lgb.LGBMClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        verbosity=-1
    )
    clf_lgb.fit(X_tr, y_tr)

    preds = clf_lgb.predict(X_vl)
    probs = clf_lgb.predict_proba(X_vl)[:, 1]

    f1 = f1_score(y_vl, preds)
    auc = roc_auc_score(y_vl, probs)
    lgb_f1_scores.append(f1)
    lgb_auc_scores.append(auc)

    print(f"Fold {fold}: F1={f1:.4f}, AUC={auc:.4f}")

print(f"\nMean F1: {np.mean(lgb_f1_scores):.4f} (+/- {np.std(lgb_f1_scores):.4f})")
print(f"Mean AUC: {np.mean(lgb_auc_scores):.4f} (+/- {np.std(lgb_auc_scores):.4f})")

Fold 0: F1=0.8261, AUC=0.9980
Fold 1: F1=0.8800, AUC=0.9959
Fold 2: F1=0.8085, AUC=0.9548
Fold 3: F1=0.7727, AUC=0.9964
Fold 4: F1=0.8800, AUC=0.9613

Mean F1: 0.8335 (+/- 0.0417)
Mean AUC: 0.9813 (+/- 0.0191)


In [6]:
from sklearn.metrics import precision_recall_curve

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_class.iloc[train_idx], y_class.iloc[val_idx]

    clf_lgb = lgb.LGBMClassifier(
        n_estimators=300, random_state=42, class_weight="balanced", verbosity=-1
    )
    clf_lgb.fit(X_tr, y_tr)
    probs = clf_lgb.predict_proba(X_vl)[:, 1]

    precisions, recalls, thresholds = precision_recall_curve(y_vl, probs)
    f1s = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
    best_idx = f1s.argmax()
    print(f"Fold {fold}: best threshold={thresholds[best_idx]:.3f}, best F1={f1s[best_idx]:.4f} (default 0.5 F1 was {lgb_f1_scores[fold]:.4f})")

Fold 0: best threshold=0.006, best F1=0.9630 (default 0.5 F1 was 0.8261)
Fold 1: best threshold=0.021, best F1=0.9455 (default 0.5 F1 was 0.8800)
Fold 2: best threshold=0.109, best F1=0.8846 (default 0.5 F1 was 0.8085)
Fold 3: best threshold=0.006, best F1=0.9630 (default 0.5 F1 was 0.7727)
Fold 4: best threshold=0.138, best F1=0.8846 (default 0.5 F1 was 0.8800)


##  This to test for the Regression


In [8]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

lgb_reg_rmse = []
lgb_reg_mae = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_vl = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_vl = y_reg.iloc[train_idx], y_reg.iloc[val_idx]

    reg_lgb = lgb.LGBMRegressor(
        n_estimators=300,
        random_state=42,
        verbosity=-1
    )
    reg_lgb.fit(X_tr, y_tr)

    preds = reg_lgb.predict(X_vl)

    rmse = np.sqrt(mean_squared_error(y_vl, preds))
    mae = mean_absolute_error(y_vl, preds)
    lgb_reg_rmse.append(rmse)
    lgb_reg_mae.append(mae)

    print(f"Fold {fold}: RMSE={rmse:.4f}, MAE={mae:.4f}")

print(f"\nMean RMSE: {np.mean(lgb_reg_rmse):.4f} (+/- {np.std(lgb_reg_rmse):.4f})")
print(f"Mean MAE: {np.mean(lgb_reg_mae):.4f} (+/- {np.std(lgb_reg_mae):.4f})")

Fold 0: RMSE=1.1039, MAE=0.6956
Fold 1: RMSE=2.8000, MAE=0.9600
Fold 2: RMSE=1.8815, MAE=0.9732
Fold 3: RMSE=2.8665, MAE=1.0619
Fold 4: RMSE=2.7493, MAE=1.2159

Mean RMSE: 2.2802 (+/- 0.6894)
Mean MAE: 0.9813 (+/- 0.1695)
